In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

def volatility_surface(start_date="2015-01-01"):

    tickers = ["SPY", "^VIX", "^VVIX"]

    raw_data = yf.download(tickers, start=start_date)['Close']
    data = raw_data.ffill().dropna()

    return data

def volatility_signals(vol_df, rv=20):

    signals = pd.DataFrame(index=vol_df.index)

    spy_returns = vol_df['SPY'].pct_change()

    realized_vol = spy_returns.rolling(window=rv).std() * np.sqrt(252) * 100
    signals['Realized_Vol'] = realized_vol
    signals['IV_VIX'] = vol_df['^VIX']

    signals['VRP'] = signals['IV_VIX'] - signals['Realized_Vol']

    vvix = vol_df['^VVIX']

    vvix_mean = vvix.rolling(window=rv).mean()
    vvix_std = vvix.rolling(window=rv).std()
    signals['VVIX_Z_Score'] = (vvix - vvix_mean) / vvix_std

    signals['Buy_Cheap'] = (signals['VRP'] < 0) & (signals['VVIX_Z_Score'] < 0)

    signals['Crisis_Alpha'] = signals['VVIX_Z_Score'] > 2.0

    return signals.dropna()

if __name__ == "__main__":
    vol_data = volatility_surface()
    vol_signals = volatility_signals(vol_data)
    
    print("\nRecent Tail-Risk Signals:")
    crisis_days = vol_signals[vol_signals['Crisis_Alpha'] == True]
    print(crisis_days[['IV_VIX', 'VRP', 'VVIX_Z_Score', 'Crisis_Alpha']].tail())

[*********************100%***********************]  3 of 3 completed



Recent Tail-Risk Signals:
            IV_VIX        VRP  VVIX_Z_Score  Crisis_Alpha
Date                                                     
2025-11-20   26.42  12.286224      2.444105          True
2026-01-14   16.75   8.118036      2.259169          True
2026-01-20   20.09   9.709556      2.820662          True
2026-02-05   21.77  10.408763      2.121639          True
2026-03-06   29.49  16.183035      3.375850          True


In [7]:
from scipy.stats import norm

def bs_put(S, K, T, r, sigma):

    if T <= 0:
        return max(0.0, K - S), -1.0 if S < K else 0.0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    put_price = (K * np.exp(-r * T) * norm.cdf(-d2)) - (S * norm.cdf(-d1))
    put_delta = norm.cdf(d1) - 1.0

    return put_price, put_delta

def risk_hedge(portfolio_value, S, K, T, r, sigma):
    
    put_price, put_delta = bs_put(S, K, T, r, sigma)

    print(f"Current Underlying Price (S): ${S:.2f}")
    print(f"Target Strike Price (K):      ${K:.2f}")
    print(f"Theoretical Put Price:        ${put_price:.2f}")
    print(f"Option Delta (\u0394):            {put_delta:.4f}")

    portfolio_equivalent_shares = portfolio_value / S

    if put_delta == 0:
        print("Delta is 0. Cannot hedge with this option.")
        return 0, 0

    hedge_ratio = 0.25
    contracts_needed = (portfolio_value * hedge_ratio) / (S * abs(put_delta) * 100)
    contracts_buy = np.ceil(contracts_needed)

    premium_cost = contracts_buy * put_price * 100
    percentage = premium_cost / portfolio_value

    print(f"Portfolio Value:            ${portfolio_value:,.2f}")
    print(f"Contracts to Purchase:      {int(contracts_buy)}")
    print(f"Total Premium Cost:         ${premium_cost:,.2f}")
    print(f"Insurance Cost Drag:        {percentage:.2%}")

    return contracts_buy, premium_cost

if __name__ == "__main__":

    portfolio_size = 500000
    spy_price = 500.00
    k = 400.00 
    expiry = 0.25  
    r= 0.05 
    vix = 0.20      

    contracts, cost = risk_hedge(
        portfolio_size,
        spy_price,
        k,
        expiry,
        r,
        vix
    )

Current Underlying Price (S): $500.00
Target Strike Price (K):      $400.00
Theoretical Put Price:        $0.14
Option Delta (Δ):            -0.0081
Portfolio Value:            $500,000.00
Contracts to Purchase:      311
Total Premium Cost:         $4,279.74
Insurance Cost Drag:        0.86%
